# Multi-provider calls, sampling controls, and temperature
**The messages format · temperature, top_p, max_tokens, stop · switching provider by editing .env**

Run top to bottom. Before starting: `git pull`, `uv sync` (no new packages today).

Two providers need a key: Groq (already in your `.env`) and Gemini (new today — see section 1). Ollama is optional; cells that need it skip cleanly if it is not running.

## 1. Setup — three clients, one function

A provider is a place; a model is a brain. `ask()` in `providers_utils.py` is the one function every provider boils down to: build a messages list, call `chat.completions.create`, read the text and the usage. Only the client and the model name change between providers.

**Getting a free Gemini key (2 minutes):** go to aistudio.google.com, sign in, click "Get API key", create one, and paste it into `.env` as `GEMINI_API_KEY`. No card needed.

In [ ]:
import sys; sys.path[:0] = [".", "module01", "../module01"]
import os
from providers_utils import ask, classify_ticket, gemini_client, groq_client, list_gemini_models, ollama_client, pick_gemini_model
from tickets_data import ALL_TICKETS, BORDERLINE_TICKETS, CLEAR_TICKETS

groq = groq_client()
GROQ_MODEL = os.getenv("MODEL")

try:
    gemini = gemini_client()
    # Gemini's Flash model name changes often - verify instead of trusting a name written down months ago.
    flash_models = list_gemini_models()
    GEMINI_MODEL = os.getenv("GEMINI_MODEL") or pick_gemini_model(flash_models)
    print("Gemini models available:", flash_models)
    print("using:", GEMINI_MODEL)
except Exception as e:
    gemini, GEMINI_MODEL = None, None
    print("Gemini not configured yet:", e)

try:
    ollama = ollama_client()
    OLLAMA_MODEL = "llama3.2:3b"
except Exception as e:
    ollama = None
    print("Ollama not available:", e)


## 2. The messages format, properly

You have used this shape since `hello.py`: a list of role/content pairs. Two things worth seeing deliberately.

- The **system** message sets who the model is. Drop it and the model falls back to a generic default persona.
- **Order and role matter** — a system instruction buried as a second user message is weaker than a real system message.

In [ ]:
system = "You are a terse assistant for a retail bank. Answer in one short sentence."
user = "What is an EMI?"

with_system = ask(groq, GROQ_MODEL, system, user)
without_system = ask(groq, GROQ_MODEL, "", user)

print("WITH system message:\n ", with_system["text"])
print("\nWITHOUT a system message:\n ", without_system["text"])


*Question 1 — What changed between the two answers, and which field in `.env` would you edit to make this run against Gemini instead of Groq?*

## 3. Same prompt, three providers

Same system prompt, same question, three providers. Compare the text and the token usage. This is the entire point of today: the code around this cell does not change — only which client and model name go in.

In [ ]:
prompt = "In one sentence, what is the difference between a workflow and an agent?"
results = {}
for label, client, model in [("groq", groq, GROQ_MODEL), ("gemini", gemini, GEMINI_MODEL), ("ollama", ollama, "llama3.2:3b" if ollama else None)]:
    if client is None or model is None:
        print(f"--- {label}: skipped (not configured)\n"); continue
    try:
        r = ask(client, model, "You are a concise AI engineering tutor.", prompt)
        results[label] = r
        print(f"--- {label} ({model})")
        print(" ", r["text"].strip())
        print(f"   tokens: prompt={r['prompt_tokens']} completion={r['completion_tokens']} total={r['total_tokens']}\n")
    except Exception as e:
        print(f"--- {label}: error ({type(e).__name__}: {str(e)[:100]})\n")

*Question 2 — Which provider used the fewest total tokens for the same question? Does that necessarily mean it is the cheapest or the best choice — what else would you need to know (hint: Session 10)?*

## 4. Sampling: temperature, top_p, max_tokens, stop

- **temperature** (0 to roughly 2) controls how much randomness goes into picking the next token. 0 is as close to deterministic as a provider allows; higher values let the model pick less-likely tokens more often.
- **top_p** is an alternative way to control the same thing — restrict the choice to the smallest set of tokens whose probabilities add up to `top_p`. Most labs here use temperature alone; the two are rarely combined.
- **max_tokens** caps the length of the reply. Set it too low on a reasoning model and you can get an empty answer, as you saw in Session 10.
- **stop** ends generation the moment a given string appears — useful for forcing a short, structured answer.

In [ ]:
short = ask(groq, GROQ_MODEL, "You are a helpful assistant.", "Explain what a vector database is.", max_tokens=15)
enough = ask(groq, GROQ_MODEL, "You are a helpful assistant.", "Explain what a vector database is.", max_tokens=150)
print("max_tokens=15:\n ", short["text"], "\n")
print("max_tokens=150:\n ", enough["text"][:300])


In [ ]:
r = ask(groq, GROQ_MODEL, "You are a helpful assistant. Answer in a short paragraph.",
        "List three benefits of retrieval-augmented generation.", max_tokens=200, stop=["\n"])
print(r["text"])


## 5. The temperature sweep — where it actually matters

Three clear tickets, three borderline ones. Run the same ticket through the classifier at several temperatures and watch what happens. A routing decision like this should usually be made at temperature 0 — you do not want "billing" vs "technical" to depend on a coin flip.

In [ ]:
TEMPERATURES = [0.0, 0.3, 0.7, 1.0]
RUNS_PER_TEMPERATURE = 3

if groq is None:
    print("skipped (no Groq client)")
else:
    for tid, text in BORDERLINE_TICKETS:
        print(f"\n=== {tid} ===\n{text}\n")
        for temp in TEMPERATURES:
            labels = []
            for _ in range(RUNS_PER_TEMPERATURE):
                result = classify_ticket(groq, GROQ_MODEL, text, temperature=temp)
                labels.append(result["label"] if result["valid"] else f"INVALID:{result['label']!r}")
            stable = len(set(labels)) == 1
            print(f"  temperature={temp:<4} {'stable' if stable else 'DRIFTED':8} {labels}")


*Question 3 — At which temperature, if any, did the label start to change between runs on the same ticket? What would you set temperature to for a routing decision in production, and why?*
*Question 4 — Run the same sweep on one of the CLEAR_TICKETS. Does it drift as much as the borderline ones? What does that tell you about when temperature actually matters?*

## 6. Switching provider — three lines, no code change

This is the payoff. Open `.env`. Comment the three Groq lines, uncomment the three Gemini (or Ollama) lines — or just point `BASE_URL`/`API_KEY`/`MODEL` at a different provider. Nothing in `providers_utils.py` or this notebook changes. Re-run the cell in section 3 after switching and compare.

In [ ]:
# After editing .env, restart the kernel (Kernel > Restart), then re-run section 1's setup cell
# and this cell to see the SAME code now talking to a different provider.
import importlib, providers_utils
importlib.reload(providers_utils)
from providers_utils import ask
print("BASE_URL now:", os.getenv("BASE_URL"))
print("MODEL now:", os.getenv("MODEL"))

## 7. Reflection

Answer questions 1–4 above, one or two lines each.

1. Ans ---> With the system message the model gave one short banking sentence. Without it the output was only **: the reply was cut off because gpt-oss spends its 200-token budget thinking first. To switch to Gemini, edit BASE_URL in .env, together with API_KEY and MODEL.

2. Ans ---> Only Groq returned numbers (97 in + 139 out = 236 total). Gemini returned empty text and Ollama's model wasn't found, so I can't compare yet. Fewest tokens doesn't mean cheapest or best: prices per million tokens differ, output costs more than input, hidden thinking is billed, tokenizers count differently, and quality, latency and data rules matter too.

3. Ans ---> The label never flipped between billing and technical. Runs were stable at 0.0 and 0.3. Drift first appeared at 0.7 (borderline_2) and again at 1.0, but as empty answers (INVALID ''), not different labels. For production routing I'd use temperature 0, check that the label is valid, and allow enough max_tokens.

4. Ans ---> (Run the sweep on CLEAR_TICKETS and write what you see.) Expected: clear tickets keep the same label. Temperature only matters when two labels are close.


Then run the checkpoint from the terminal: `uv run pytest tests/test_s11.py`